# QuTiP Interoperability and Validation

This notebook validates the current NumPy-based `pec` implementation against QuTiP while keeping QuTiP strictly optional for the main package. It demonstrates state conversion, metric agreement, Bell-state analysis on QuTiP-backed states, and a small timing comparison for representative cases.

## Scope

The goal here is interoperability and validation rather than replacement. The package state construction and analysis remain in `pec.states`, `pec.metrics`, and `pec.bell`; QuTiP is used as an external reference and object model through `pec.qutip_tools`.

One important convention note: QuTiP's raw fidelity function uses a square-root convention, while `pec.metrics.fidelity` reports the squared density-matrix fidelity. The helper `pec.qutip_tools.qutip_fidelity(...)` squares the QuTiP result so the two libraries are compared on the same definition.

In [1]:
from pathlib import Path
import sys

search_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_roots if (path / "src" / "pec").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate the repository root containing src/pec.")

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Repository root: {repo_root}")
print(f"Using package source: {src_path}")

Repository root: c:\Users\omaru\photonic-entanglement-characterization
Using package source: c:\Users\omaru\photonic-entanglement-characterization\src


In [3]:
import numpy as np
import pandas as pd
from IPython.display import display
from time import perf_counter

try:
    import qutip
except ImportError as exc:
    raise ImportError("This notebook requires the optional QuTiP dependency. Install `pec[qutip]` to run it.") from exc

from pec import bell
from pec import metrics
from pec import states
import pec.qutip_tools as qutip_tools

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = "{:.10f}".format

print(f"QuTiP version: {qutip.__version__}")

QuTiP version: 5.2.2


In [4]:
import sys
print(sys.executable)

import pec
import qutip
print("pec ok")
print("qutip ok")
print(qutip.__version__)

C:\Users\omaru\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
pec ok
qutip ok
5.2.2


## 1. Build representative PEC states

We begin with a few standard single-qubit and two-qubit states from `pec.states`, including a mixed Bell-like density matrix that is useful for fidelity and trace-distance comparisons.

In [5]:
ket_h = states.ket_h()
ket_r = states.ket_r()
ket_phi_plus = states.bell_state("phi_plus")

rho_h = states.density_matrix(ket_h)
rho_r = states.density_matrix(ket_r)
rho_phi_plus = states.density_matrix(ket_phi_plus)
rho_mixed = states.trace_normalize(0.82 * rho_phi_plus + 0.18 * states.tensor_projector("HV"))

state_table = pd.DataFrame(
    {
        "Object": ["|H>", "|R>", "|Phi+>", "rho_mixed"],
        "Kind": ["ket", "ket", "ket", "density matrix"],
        "Shape": [ket_h.shape, ket_r.shape, ket_phi_plus.shape, rho_mixed.shape],
    }
).set_index("Object")

state_table

,Kind,Shape
Object,,
|H>,ket,"(2,)"
|R>,ket,"(2,)"
|Phi+>,ket,"(4,)"
rho_mixed,density matrix,"(4, 4)"


## 2. Convert PEC objects to QuTiP `Qobj`

The interoperability layer accepts plain NumPy kets and operators, and it can preserve subsystem structure through the optional `dims` argument.

In [6]:
qobj_h = qutip_tools.to_qobj(ket_h, dims=[2])
qobj_phi_plus = qutip_tools.to_qobj(ket_phi_plus, dims=[2, 2])
qobj_rho_mixed = qutip_tools.to_qobj(rho_mixed, dims=[2, 2])

qobj_table = pd.DataFrame(
    {
        "Object": ["|H>", "|Phi+>", "rho_mixed"],
        "Type": ["ket", "ket", "operator"],
        "Qobj shape": [qobj_h.shape, qobj_phi_plus.shape, qobj_rho_mixed.shape],
        "dims": [qobj_h.dims, qobj_phi_plus.dims, qobj_rho_mixed.dims],
        "Round-trip matches PEC": [
            np.allclose(qutip_tools.from_qobj(qobj_h), ket_h),
            np.allclose(qutip_tools.from_qobj(qobj_phi_plus), ket_phi_plus),
            np.allclose(qutip_tools.from_qobj(qobj_rho_mixed), rho_mixed),
        ],
    }
).set_index("Object")

qobj_table

,Type,Qobj shape,dims,Round-trip matches PEC
Object,,,,
|H>,ket,"(2, 1)","[[2], [1]]",True
|Phi+>,ket,"(4, 1)","[[2, 2], [1]]",True
rho_mixed,operator,"(4, 4)","[[2, 2], [2, 2]]",True


## 3. Compare core metrics against QuTiP

The helper `compare_metric_values(...)` reports side-by-side values from `pec.metrics` and QuTiP, together with absolute differences and a tolerance check.

In [7]:
comparison_cases = [
    ("rho_H", rho_h, None),
    ("rho_Phi+", rho_phi_plus, None),
    ("rho_mixed", rho_mixed, None),
    ("rho_R vs rho_H", rho_r, rho_h),
    ("rho_mixed vs rho_Phi+", rho_mixed, rho_phi_plus),
]

comparison_rows = []
for case_name, a, b in comparison_cases:
    comparison = qutip_tools.compare_metric_values(a, b)
    for metric_name, values in comparison.items():
        comparison_rows.append(
            {
                "Case": case_name,
                "Metric": metric_name,
                "PEC": values["pec"],
                "QuTiP": values["qutip"],
                "Absolute difference": values["abs_diff"],
                "Within tolerance": values["within_tolerance"],
            }
        )

comparison_table = pd.DataFrame(comparison_rows)
comparison_table

,Case,Metric,PEC,QuTiP,Absolute difference,Within tolerance
0,rho_H,purity,1.0000000000,1.0000000000,0.0000000000,True
1,rho_Phi+,purity,1.0000000000,1.0000000000,0.0000000000,True
2,rho_mixed,purity,0.7048000000,0.7048000000,0.0000000000,True
3,rho_R vs rho_H,purity,1.0000000000,1.0000000000,0.0000000000,True
4,rho_R vs rho_H,fidelity,0.5000000000,0.5000000000,0.0000000000,True
5,rho_R vs rho_H,trace_distance,0.7071067812,0.7071067812,0.0000000000,True
6,rho_mixed vs rho_Phi+,purity,0.7048000000,0.7048000000,0.0000000000,True
7,rho_mixed vs rho_Phi+,fidelity,0.8200000000,0.8200000000,0.0000000000,True
8,rho_mixed vs rho_Phi+,trace_distance,0.1800000000,0.1800000007,0.0000000007,True


## 4. Expectation values with QuTiP-backed operators

The interoperability layer also provides a small expectation-value helper. For the ideal `|Phi+>` state, the `Z \otimes Z` correlator should be `+1`.

In [8]:
zz_operator = np.kron(states.pauli("Z"), states.pauli("Z"))
zz_expectation = qutip_tools.qutip_expect(zz_operator, rho_phi_plus)
zz_reference = bell.two_qubit_correlation(rho_phi_plus, [0.0, 0.0, 1.0], [0.0, 0.0, 1.0])

pd.DataFrame(
    {
        "Quantity": ["QuTiP expectation", "PEC Bell correlator"],
        "Value": [zz_expectation, zz_reference],
    }
).set_index("Quantity")

,Value
Quantity,
QuTiP expectation,1.0000000000
PEC Bell correlator,1.0000000000


## 5. Bell-state fidelities for QuTiP-backed states

The Bell-state fidelity calculation remains our own NumPy-based analysis. Here we use QuTiP as the underlying state representation and convert back to dense arrays only at the point where the PEC Bell helper is applied.

In [9]:
qutip_backed_states = {
    "Ideal Phi+": qutip_tools.to_qobj(rho_phi_plus, dims=[2, 2]),
    "Mixed Bell-like": qutip_tools.to_qobj(rho_mixed, dims=[2, 2]),
}

bell_rows = []
for label, q_state in qutip_backed_states.items():
    fidelities = bell.bell_state_fidelities(qutip_tools.from_qobj(q_state))
    bell_rows.append(
        {
            "State": label,
            "Phi+": fidelities["phi_plus"],
            "Phi-": fidelities["phi_minus"],
            "Psi+": fidelities["psi_plus"],
            "Psi-": fidelities["psi_minus"],
        }
    )

bell_fidelity_table = pd.DataFrame(bell_rows).set_index("State")
bell_fidelity_table

,Phi+,Phi-,Psi+,Psi-
State,,,,
Ideal Phi+,1.0000000000,0.0000000000,0.0000000000,0.0000000000
Mixed Bell-like,0.8200000000,0.0000000000,0.0900000000,0.0900000000


## 6. Small timing comparison

The timings below compare the current PEC NumPy implementation against the QuTiP-backed validation path in `pec.qutip_tools`. These QuTiP timings include NumPy-to-`Qobj` conversion overhead, which is often the relevant practical cost when QuTiP is used as an interoperability layer rather than the package's native internal representation.

In [10]:
def average_runtime(function, *args, repeats=250):
    start = perf_counter()
    for _ in range(repeats):
        function(*args)
    return (perf_counter() - start) / repeats * 1e6


timing_specs = [
    ("purity(rho_H)", metrics.purity, qutip_tools.qutip_purity, (rho_h,)),
    ("purity(rho_mixed)", metrics.purity, qutip_tools.qutip_purity, (rho_mixed,)),
    ("fidelity(rho_mixed, rho_Phi+)", metrics.fidelity, qutip_tools.qutip_fidelity, (rho_mixed, rho_phi_plus)),
    (
        "trace_distance(rho_mixed, rho_Phi+)",
        metrics.trace_distance,
        qutip_tools.qutip_trace_distance,
        (rho_mixed, rho_phi_plus),
    ),
]

timing_rows = []
for label, pec_fn, qutip_fn, args in timing_specs:
    timing_rows.append(
        {
            "Operation": label,
            "PEC time (us)": average_runtime(pec_fn, *args),
            "QuTiP time (us)": average_runtime(qutip_fn, *args),
        }
    )

timing_table = pd.DataFrame(timing_rows)
timing_table["QuTiP / PEC"] = timing_table["QuTiP time (us)"] / timing_table["PEC time (us)"]
timing_table

,Operation,PEC time (us),QuTiP time (us),QuTiP / PEC
0,purity(rho_H),6.1116000288,11.3220000057,1.8525426979
1,purity(rho_mixed),5.4175999830,10.8652000199,2.0055375173
2,"fidelity(rho_mixed, rho_Phi+)",40.2744000312,171.3279999676,4.2540174363
3,"trace_distance(rho_mixed, rho_Phi+)",8.1931999885,62.4272000277,7.6193917048


## Summary

In typical use, the agreement table above should show machine-precision consistency between the PEC metric implementations and the QuTiP validation layer. The practical tradeoff is straightforward: the native PEC NumPy path is lighter and usually faster for this repository's current workflow, while QuTiP provides a well-established external reference together with richer object metadata such as subsystem dimensions. That makes `pec.qutip_tools` a useful optional layer for validation, interoperability, and future cross-checking without changing the core package design.